# Clean Preprocessing Pipeline

Organized version of `preprocessing3.ipynb`. Run cells from top to bottom.

In [3]:
# Colab install cell (run once)
!pip install newspaper3k feedparser requests python-dateutil spacy nltk edgartools lxml_html_clean -q
!python -m spacy download en_core_web_sm -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 74.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart P

In [4]:
"""
clean_preprocessing.py

Organized preprocessing pipeline for the Financial News RAG Capstone.

Main idea:
    User question
    -> detect company and time range
    -> retrieve candidate news articles from NewsAPI and RSS
    -> scrape full article text
    -> filter low-quality / irrelevant articles
    -> optionally retrieve SEC filings
    -> clean and chunk all documents
    -> return clean dataframes ready for RAG / FinBERT

Recommended Colab install:
    !pip install newspaper3k feedparser requests python-dateutil spacy nltk edgartools lxml_html_clean -q
    !python -m spacy download en_core_web_sm -q
"""

from __future__ import annotations

import os
import re
import logging
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional, Tuple

import pandas as pd
import requests
import feedparser
from dateutil import parser as dateutil_parser
from newspaper import Article

# Optional imports. Only needed if include_edgar=True.
try:
    from edgar import Company, set_identity
except Exception:
    Company = None
    set_identity = None

# Optional imports for chunking.
try:
    import nltk
    from nltk.tokenize import sent_tokenize
    nltk.download("punkt", quiet=True)
except Exception:
    sent_tokenize = None

try:
    import spacy
except Exception:
    spacy = None

In [5]:
# =============================================================================
# 0. Config
# =============================================================================

NEWSAPI_URL = "https://newsapi.org/v2/everything"

NEWSAPI_DOMAINS = ",".join([
    "reuters.com",
    "apnews.com",
    "cnbc.com",
    "marketwatch.com",
    "finance.yahoo.com",
    "thestreet.com",
    "nasdaq.com",
    "fortune.com",
    "fool.com",
    "businesswire.com",
    "prnewswire.com",
    "techcrunch.com",
    "venturebeat.com",
    "wired.com",
    "arstechnica.com",
    "zdnet.com",
    "theguardian.com",
])

RSS_FEEDS = [
    ("Reuters Business",      "https://feeds.reuters.com/reuters/businessNews"),
    ("Reuters Technology",    "https://feeds.reuters.com/reuters/technologyNews"),
    ("AP News Business",      "https://feeds.apnews.com/rss/apf-business"),
    ("CNBC Top News",         "https://feeds.content.dowjones.io/public/rss/mktgNewsHeadlines"),
    ("CNBC Technology",       "https://feeds.content.dowjones.io/public/rss/mktgTechNewsHeadlines"),
    ("MarketWatch",           "https://feeds.marketwatch.com/marketwatch/topstories"),
    ("The Motley Fool",       "https://www.fool.com/feeds/index.aspx"),
    ("The Street",            "https://www.thestreet.com/rss/index.xml"),
    ("TechCrunch",            "https://techcrunch.com/category/startups/feed/"),
    ("VentureBeat",           "https://venturebeat.com/feed/"),
    ("Wired Business",        "https://www.wired.com/feed/category/business/latest/rss"),
    ("Ars Technica",          "https://feeds.arstechnica.com/arstechnica/index"),
    ("PR Newswire Tech",      "https://www.prnewswire.com/rss/news-releases-list.rss"),
    ("Business Wire",         "https://feed.businesswire.com/rss/home/?rss=G1"),
    ("The Guardian Business", "https://www.theguardian.com/business/rss"),
    ("The Guardian Tech",     "https://www.theguardian.com/technology/rss"),
    ("BBC Business",          "https://feeds.bbci.co.uk/news/business/rss.xml"),
    ("Fortune",               "https://fortune.com/feed"),
]

SOURCE_PRIORITY = {
    "reuters":          10,
    "associated press": 10,
    "ap news":          10,
    "cnbc":             9,
    "marketwatch":      9,
    "yahoo finance":    8,
    "nasdaq":           8,
    "fortune":          8,
    "the street":       7,
    "thestreet":        7,
    "the guardian":     7,
    "business wire":    7,
    "businesswire":     7,
    "pr newswire":      7,
    "prnewswire":       7,
    "techcrunch":       6,
    "venturebeat":      6,
    "wired":            6,
    "ars technica":     6,
    "arstechnica":      6,
    "zdnet":            5,
    "motley fool":      5,
}

ALIAS_MAP = {
    "meta":       {"meta platforms", "facebook", "META"},
    "apple":      {"aapl", "AAPL"},
    "google":     {"alphabet", "googl", "GOOGL", "GOOG"},
    "amazon":     {"amzn", "AMZN"},
    "tesla":      {"tsla", "TSLA"},
    "nvidia":     {"nvda", "NVDA"},
    "microsoft":  {"msft", "MSFT"},
    "netflix":    {"nflx", "NFLX"},
    "uber":       {"uber technologies"},
    "twitter":    {"x corp", "x.com"},
    "snapchat":   {"snap inc", "snap"},
    "spotify":    {"spot"},
    "airbnb":     {"abnb"},
    "lyft":       {"lyft inc"},
    "samsung":    {"samsung electronics"},
    "intel":      {"intc", "INTC"},
    "amd":        {"advanced micro devices", "AMD"},
    "salesforce": {"crm", "CRM"},
    "oracle":     {"orcl", "ORCL"},
    "ibm":        {"international business machines", "IBM"},
}

BOILERPLATE_PATTERNS = [
    r"subscribe to our newsletter",
    r"sign up for",
    r"enable javascript",
    r"cookies? to improve",
    r"please enable",
    r"advertisement",
]

HEADER_NOISE_PATTERNS = [
    r"add .+ as your preferred",
    r"^share$",
    r"^read more$",
    r"^advertisement$",
    r"^skip to",
    r"^sign in$",
    r"^subscribe$",
    r"^follow us",
    r"^newsletter$",
]

STRIP_PATTERNS_EARNINGS = [
    r"(?i)forward.looking statements.{0,2000}",
    r"(?i)safe harbor.{0,1000}",
    r"(?i)private securities litigation.{0,500}",
    r"(?i)(investor relations|media contact|for more information).{0,300}",
    r"(?i)^(document|exhibit\s*\d+\.\d+)\s*$",
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "Connection":      "keep-alive",
}

MAX_WORKERS = 10
MAX_PER_SOURCE = 7
PAGE_SIZE = 100
MIN_TEXT_LEN = 200

MAX_TOKENS = 400
OVERLAP_SENTENCES = 1
MIN_CHUNK_CHARS = 100
AVG_CHARS_PER_TOKEN = 4

In [6]:
# =============================================================================
# 1. Logging and data models
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


@dataclass
class CandidateArticle:
    title: str
    source: str
    published_at: datetime
    url: str
    raw_text: str = ""
    drop_reason: str = ""


@dataclass
class Chunk:
    text: str
    company: str
    ticker: str
    source: str
    source_type: str
    published_at: str
    url: str
    chunk_index: int
    chunk_total: int
    char_count: int = field(init=False)
    token_count: int = field(init=False)

    def __post_init__(self):
        self.char_count = len(self.text)
        self.token_count = max(1, len(self.text) // AVG_CHARS_PER_TOKEN)

In [7]:
# =============================================================================
# 2. NER and query parsing
# =============================================================================

_NLP = None

def get_nlp():
    """Load spaCy only once."""
    global _NLP
    if _NLP is None:
        if spacy is None:
            raise ImportError("spaCy is not installed. Run: pip install spacy")
        _NLP = spacy.load("en_core_web_sm")
    return _NLP


def detect_company(question: str) -> str:
    """
    Extract company name from the user question.
    Example: 'What is happening with Meta lately?' -> 'Meta'
    """
    nlp = get_nlp()
    doc = nlp(question)

    orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    if orgs:
        return orgs[0]

    skip = {"what", "who", "how", "when", "where", "why", "is", "are", "the", "a", "an"}
    for token in doc:
        if token.text and token.text[0].isupper() and token.text.lower() not in skip:
            return token.text

    return ""


def detect_days_back(question: str, default_days: int = 7, max_days: int = 25) -> int:
    """
    Convert time expression in question into an integer number of days.
    Free NewsAPI plans often limit historical range, so max_days defaults to 25.
    """
    q = question.lower()
    patterns = [
        (r"\btoday\b|\bright now\b|\bcurrently\b", 1),
        (r"\bthis week\b|\blately\b|\brecently\b", 7),
        (r"\blast week\b", 7),
        (r"\bthis month\b|\blast month\b", 30),
        (r"\blast (\d+) days?\b", "days"),
        (r"\blast (\d+) weeks?\b", "weeks"),
        (r"\blast (\d+) months?\b", "months"),
    ]

    for pattern, unit in patterns:
        m = re.search(pattern, q)
        if not m:
            continue

        if isinstance(unit, int):
            return min(unit, max_days)

        n = int(m.group(1))
        if unit == "weeks":
            days = n * 7
        elif unit == "months":
            days = n * 30
        else:
            days = n

        return min(days, max_days)

    return min(default_days, max_days)


def build_search_query(company: str) -> str:
    """Build a search string for NewsAPI."""
    return f'"{company}"'


def get_company_aliases(company: str) -> set:
    company_lower = company.lower()
    aliases = {company_lower}
    aliases.update({a.lower() for a in ALIAS_MAP.get(company_lower, set())})
    return aliases

In [8]:
# =============================================================================
# 3. News retrieval
# =============================================================================

def get_source_priority(source: str) -> int:
    source_lower = source.lower()
    return max((v for k, v in SOURCE_PRIORITY.items() if k in source_lower), default=3)


def fetch_newsapi(
    company: str,
    days_back: int,
    newsapi_key: Optional[str] = None,
    page_size: int = PAGE_SIZE,
) -> List[CandidateArticle]:
    """
    Retrieve article metadata from NewsAPI.
    Output contains title/source/date/url only. Full text is scraped later.
    """
    api_key = newsapi_key or os.getenv("NEWSAPI_KEY")
    if not api_key:
        log.warning("No NEWSAPI_KEY found. Skipping NewsAPI retrieval.")
        return []

    from_date = (datetime.now(timezone.utc) - timedelta(days=days_back)).strftime("%Y-%m-%d")
    seen_urls = set()
    articles = []

    query_variants = [
        {
            "q": build_search_query(company),
            "domains": NEWSAPI_DOMAINS,
            "pageSize": page_size,
        },
        {
            "q": f'{build_search_query(company)} site:finance.yahoo.com',
            "pageSize": 20,
        },
    ]

    for params_extra in query_variants:
        params = {
            "from": from_date,
            "sortBy": "relevancy",
            "language": "en",
            "apiKey": api_key,
            **params_extra,
        }

        try:
            resp = requests.get(NEWSAPI_URL, params=params, timeout=15)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            log.warning(f"NewsAPI request failed: {e}")
            continue

        if data.get("status") != "ok":
            log.warning(f"NewsAPI returned non-ok status: {data.get('message')}")
            continue

        for item in data.get("articles", []):
            url = item.get("url", "")
            if not url or url in seen_urls:
                continue

            seen_urls.add(url)
            try:
                pub = dateutil_parser.parse(item["publishedAt"]).replace(tzinfo=timezone.utc)
            except Exception:
                pub = datetime.now(timezone.utc)

            articles.append(CandidateArticle(
                title=item.get("title", ""),
                source=item.get("source", {}).get("name", "NewsAPI"),
                published_at=pub,
                url=url,
            ))

    log.info(f"NewsAPI: {len(articles)} candidates")
    return articles


def fetch_rss_feed(feed_name: str, feed_url: str, company: str, days_back: int) -> List[CandidateArticle]:
    cutoff = datetime.now(timezone.utc) - timedelta(days=days_back)
    aliases = get_company_aliases(company)
    articles = []

    try:
        feed = feedparser.parse(feed_url)
    except Exception as e:
        log.warning(f"RSS fetch failed for {feed_name}: {e}")
        return []

    for entry in feed.entries:
        try:
            pub_struct = entry.get("published_parsed") or entry.get("updated_parsed")
            pub = datetime(*pub_struct[:6], tzinfo=timezone.utc) if pub_struct else datetime.now(timezone.utc)
        except Exception:
            pub = datetime.now(timezone.utc)

        if pub < cutoff:
            continue

        title = entry.get("title", "")
        summary = entry.get("summary", "")
        url = entry.get("link", "")
        combined_text = f"{title} {summary}".lower()

        if not any(alias in combined_text for alias in aliases):
            continue

        articles.append(CandidateArticle(
            title=title,
            source=feed_name,
            published_at=pub,
            url=url,
        ))

    return articles


def fetch_all_rss(company: str, days_back: int) -> List[CandidateArticle]:
    all_articles = []

    with ThreadPoolExecutor(max_workers=min(len(RSS_FEEDS), MAX_WORKERS)) as executor:
        futures = {
            executor.submit(fetch_rss_feed, name, url, company, days_back): name
            for name, url in RSS_FEEDS
        }

        for future in as_completed(futures):
            try:
                all_articles.extend(future.result())
            except Exception as e:
                log.warning(f"RSS future error: {e}")

    log.info(f"RSS total: {len(all_articles)} candidates")
    return all_articles


def merge_and_deduplicate(
    newsapi_articles: List[CandidateArticle],
    rss_articles: List[CandidateArticle],
    max_per_source: int = MAX_PER_SOURCE,
) -> List[CandidateArticle]:
    """
    Merge NewsAPI and RSS candidates.
    Deduplicate by URL, then cap per source to avoid one source dominating.
    """
    seen_urls = set()
    merged = []

    for art in newsapi_articles + rss_articles:
        if art.url and art.url not in seen_urls:
            seen_urls.add(art.url)
            merged.append(art)

    merged.sort(key=lambda a: (get_source_priority(a.source), a.published_at), reverse=True)

    source_counts = defaultdict(int)
    capped = []

    for art in merged:
        if source_counts[art.source] < max_per_source:
            capped.append(art)
            source_counts[art.source] += 1

    log.info(f"Merged candidates: {len(merged)} unique -> {len(capped)} after source cap")
    return capped

In [9]:
# =============================================================================
# 4. Article scraping and filtering
# =============================================================================

def scrape_article(art: CandidateArticle) -> CandidateArticle:
    """
    Three-stage scraper:
        1. newspaper3k downloader
        2. requests + newspaper3k parser
        3. raw <p> tag extraction fallback
    """
    try:
        article = Article(art.url)
        article.download()
        article.parse()
        art.raw_text = article.text.strip()

        if len(art.raw_text) < MIN_TEXT_LEN:
            resp = requests.get(art.url, headers=HEADERS, timeout=12)
            if resp.status_code == 200:
                article2 = Article(art.url)
                article2.set_html(resp.text)
                article2.parse()
                candidate = article2.text.strip()
                if len(candidate) > len(art.raw_text):
                    art.raw_text = candidate

        if len(art.raw_text) < MIN_TEXT_LEN:
            resp = requests.get(art.url, headers=HEADERS, timeout=12)
            if resp.status_code == 200:
                paragraphs = re.findall(
                    r"<p[^>]*>(.*?)</p>",
                    resp.text,
                    re.DOTALL | re.IGNORECASE,
                )
                clean_paras = [
                    re.sub(r"<[^>]+>", "", p).strip()
                    for p in paragraphs
                ]
                body = "\n\n".join(p for p in clean_paras if len(p) > 40)
                if len(body) > len(art.raw_text):
                    art.raw_text = body

    except Exception as e:
        if not art.raw_text:
            art.drop_reason = f"scrape_error: {e}"

    return art


def parallel_scrape(candidates: List[CandidateArticle]) -> List[CandidateArticle]:
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(scrape_article, art): art for art in candidates}

        for future in as_completed(futures):
            try:
                results.append(future.result())
            except Exception as e:
                log.warning(f"Scrape future error: {e}")

    return results


def clean_first_para(text: str) -> str:
    """
    Remove header/navigation lines before checking whether the company appears.
    This prevents noise like 'Add AP News on Google' from counting as a real mention.
    """
    lines = text.split("\n")
    clean_lines = [
        line for line in lines
        if not any(re.search(pattern, line.strip().lower()) for pattern in HEADER_NOISE_PATTERNS)
    ]
    return "\n".join(clean_lines)[:500].lower()


def apply_filters(
    articles: List[CandidateArticle],
    company: str,
    min_text_len: int = MIN_TEXT_LEN,
) -> Tuple[List[CandidateArticle], List[CandidateArticle]]:
    """
    Keep only articles that:
        - have non-empty full text
        - are long enough
        - are not obvious boilerplate
        - mention the company or alias in the title or first paragraph
    """
    aliases = get_company_aliases(company)
    kept, dropped = [], []

    for art in articles:
        if art.drop_reason:
            dropped.append(art)
            continue

        if not art.raw_text.strip():
            art.drop_reason = "empty_text"
            dropped.append(art)
            continue

        if len(art.raw_text) < min_text_len:
            art.drop_reason = f"too_short ({len(art.raw_text)} chars)"
            dropped.append(art)
            continue

        lower_text = art.raw_text.lower()
        if any(re.search(pattern, lower_text) for pattern in BOILERPLATE_PATTERNS):
            art.drop_reason = "boilerplate"
            dropped.append(art)
            continue

        title_lower = art.title.lower()
        first_para = clean_first_para(art.raw_text)

        in_title = any(alias in title_lower for alias in aliases)
        in_first_para = any(alias in first_para for alias in aliases)

        if not (in_title or in_first_para):
            art.drop_reason = "not_subject"
            dropped.append(art)
            continue

        kept.append(art)

    log.info(f"Filtering: {len(kept)} kept, {len(dropped)} dropped")
    return kept, dropped


def articles_to_dataframe(
    kept: List[CandidateArticle],
    dropped: Optional[List[CandidateArticle]] = None,
    company: str = "",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Convert kept and dropped articles into clean dataframes."""
    df_news = pd.DataFrame([{
        "company": company,
        "title": art.title,
        "source": art.source,
        "published_at": art.published_at,
        "url": art.url,
        "raw_text": art.raw_text,
        "char_count": len(art.raw_text),
        "priority": get_source_priority(art.source),
    } for art in kept])

    df_debug = pd.DataFrame([{
        "company": company,
        "title": art.title,
        "source": art.source,
        "published_at": art.published_at,
        "url": art.url,
        "drop_reason": art.drop_reason,
    } for art in (dropped or [])])

    return df_news, df_debug


def retrieve_clean_news(
    question: str,
    newsapi_key: Optional[str] = None,
    days_back: Optional[int] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict]:
    """
    Main news preprocessing function.

    Returns:
        df_news: kept articles with raw_text
        df_debug: dropped articles and reasons
        metadata: company, days_back, question
    """
    company = detect_company(question)
    if not company:
        raise ValueError("Could not detect company name from question.")

    if days_back is None:
        days_back = detect_days_back(question)

    log.info(f"Question: {question}")
    log.info(f"Company: {company}")
    log.info(f"Days back: {days_back}")

    with ThreadPoolExecutor(max_workers=2) as executor:
        f_newsapi = executor.submit(fetch_newsapi, company, days_back, newsapi_key)
        f_rss = executor.submit(fetch_all_rss, company, days_back)
        newsapi_articles = f_newsapi.result()
        rss_articles = f_rss.result()

    candidates = merge_and_deduplicate(newsapi_articles, rss_articles)
    scraped = parallel_scrape(candidates)
    kept, dropped = apply_filters(scraped, company)

    df_news, df_debug = articles_to_dataframe(kept, dropped, company=company)

    metadata = {
        "question": question,
        "company": company,
        "days_back": days_back,
        "num_candidates": len(candidates),
        "num_kept": len(df_news),
        "num_dropped": len(df_debug),
    }

    return df_news, df_debug, metadata

In [10]:
# =============================================================================
# 5. Optional SEC / EDGAR preprocessing
# =============================================================================

def extract_text_from_filing(filing_obj) -> str:
    """Try multiple methods to extract text from an EDGAR filing object."""
    try:
        prs = filing_obj.press_releases
        if prs and len(prs) > 0:
            pr = prs[0]
            for attr in ["text", "markdown", "content", "html"]:
                if hasattr(pr, attr):
                    val = getattr(pr, attr)
                    if callable(val):
                        result = val()
                        if isinstance(result, str) and len(result) > 200:
                            return result[:50000]
                    elif isinstance(val, str) and len(val) > 200:
                        return val[:50000]
    except Exception:
        pass

    for method in ["text", "markdown"]:
        try:
            val = getattr(filing_obj, method)()
            if isinstance(val, str) and len(val) > 200:
                return val[:50000]
        except Exception:
            pass

    try:
        text = str(filing_obj)
        if len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    return ""


def is_earnings_filing(filing_obj) -> bool:
    """Check whether an 8-K filing likely contains earnings results."""
    try:
        items = filing_obj.items
        if any("2.02" in str(item) for item in items):
            return True
    except Exception:
        pass

    try:
        filing_str = str(filing_obj).lower()
        keywords = [
            "2.02",
            "results of operations",
            "quarterly results",
            "financial results",
            "earnings per share",
            "revenue was",
        ]
        if any(kw in filing_str for kw in keywords):
            return True
    except Exception:
        pass

    return False


def get_earnings_filings(
    ticker: str,
    company_name: str,
    sec_identity: str,
    max_earnings: int = 4,
    max_scan: int = 30,
) -> pd.DataFrame:
    """
    Pull recent earnings 8-K filings from EDGAR.
    """
    if Company is None or set_identity is None:
        raise ImportError("edgartools is not installed. Run: pip install edgartools")

    set_identity(sec_identity)
    company = Company(ticker)
    filings = company.get_filings(form="8-K").head(max_scan)

    rows = []
    for filing in filings:
        try:
            filing_date = str(filing.filing_date)
            obj = filing.obj()

            if not is_earnings_filing(obj):
                continue

            text = extract_text_from_filing(obj)
            if len(text.strip()) < 200:
                continue

            rows.append({
                "ticker": ticker.upper(),
                "company": company_name,
                "section_type": "8-K earnings",
                "filing_type": "8-K",
                "filing_date": filing_date,
                "source": "SEC EDGAR 8-K",
                "url": f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={ticker}",
                "text": text.strip(),
            })

            if len(rows) >= max_earnings:
                break

        except Exception as e:
            log.warning(f"Skipping filing {getattr(filing, 'filing_date', '')}: {e}")

    return pd.DataFrame(rows)


def get_latest_10k_sections(
    ticker: str,
    company_name: str,
    sec_identity: str,
) -> pd.DataFrame:
    """
    Pull Risk Factors and MD&A from the latest 10-K.
    """
    if Company is None or set_identity is None:
        raise ImportError("edgartools is not installed. Run: pip install edgartools")

    set_identity(sec_identity)
    company = Company(ticker)
    filing = company.get_filings(form="10-K").latest()
    tenk = filing.obj()

    filing_date = str(filing.filing_date)
    url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={ticker}"

    rows = []

    risk_text = getattr(tenk, "risk_factors", "")
    if isinstance(risk_text, str) and len(risk_text) > 200:
        rows.append({
            "ticker": ticker.upper(),
            "company": company_name,
            "section_type": "Risk Factors",
            "filing_type": "10-K",
            "filing_date": filing_date,
            "source": "SEC EDGAR 10-K",
            "url": url,
            "text": risk_text,
        })

    try:
        full_text = filing.text()
        pattern = re.compile(
            r"(item\s*7\.?\s*management.*?discussion.*?analysis.*?)(item\s*7a\.|item\s*8\.)",
            re.IGNORECASE | re.DOTALL,
        )
        candidates = []
        for match in pattern.finditer(full_text):
            section = match.group(1).strip()
            if len(section) > 2000:
                candidates.append(section)

        if candidates:
            mda_text = max(candidates, key=len)
            rows.append({
                "ticker": ticker.upper(),
                "company": company_name,
                "section_type": "MD&A",
                "filing_type": "10-K",
                "filing_date": filing_date,
                "source": "SEC EDGAR 10-K",
                "url": url,
                "text": mda_text,
            })
    except Exception as e:
        log.warning(f"Could not extract MD&A section: {e}")

    return pd.DataFrame(rows)

In [11]:
# =============================================================================
# 6. Cleaning and chunking
# =============================================================================

def clean_text(text: str, source_type: str = "news") -> str:
    """Basic text cleaning before chunking."""
    if not text:
        return ""

    text = re.sub(r"<[^>]+>", " ", str(text))
    text = re.sub(r"&[a-z]+;", " ", text)

    if source_type in {"earnings", "8-K earnings"}:
        for pattern in STRIP_PATTERNS_EARNINGS:
            text = re.sub(pattern, "", text, flags=re.DOTALL | re.MULTILINE)

        clean_lines = []
        for line in text.split("\n"):
            stripped = line.strip()
            if not stripped:
                clean_lines.append(line)
                continue

            alpha_chars = sum(1 for c in stripped if c.isalpha())
            total_chars = len(stripped)

            if total_chars == 0 or alpha_chars / total_chars >= 0.4:
                clean_lines.append(line)

        text = "\n".join(clean_lines)

    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


def semantic_chunk(
    text: str,
    max_tokens: int = MAX_TOKENS,
    overlap_sentences: int = OVERLAP_SENTENCES,
) -> List[str]:
    """Sentence-based chunking that tries not to split meaning in the middle."""
    if sent_tokenize is not None:
        try:
            sentences = sent_tokenize(text)
        except Exception:
            sentences = re.split(r"(?<=[.!?])\s+", text)
    else:
        sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    current = []
    cur_tokens = 0

    for sentence in sentences:
        sent_tokens = max(1, len(sentence) // AVG_CHARS_PER_TOKEN)

        if cur_tokens + sent_tokens > max_tokens and current:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences > 0 else []
            cur_tokens = sum(max(1, len(s) // AVG_CHARS_PER_TOKEN) for s in current)

        current.append(sentence)
        cur_tokens += sent_tokens

    if current:
        chunks.append(" ".join(current))

    return chunks


def fixed_chunk(text: str, max_tokens: int = MAX_TOKENS, overlap_chars: int = 50) -> List[str]:
    """Fallback chunker for very long text blocks."""
    max_chars = max_tokens * AVG_CHARS_PER_TOKEN
    chunks = []
    start = 0

    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end].strip())
        start = max(end - overlap_chars, start + 1)

    return [c for c in chunks if c]


def hybrid_chunk(text: str, source_type: str = "news", max_tokens: int = MAX_TOKENS) -> List[str]:
    """
    Hybrid chunking:
        - first try sentence-based chunking
        - if a chunk is still too long, split by fixed size
    """
    cleaned = clean_text(text, source_type)
    if not cleaned:
        return []

    rough_chunks = semantic_chunk(cleaned, max_tokens=max_tokens)
    final_chunks = []

    for chunk in rough_chunks:
        if len(chunk) // AVG_CHARS_PER_TOKEN > max_tokens:
            final_chunks.extend(fixed_chunk(chunk, max_tokens=max_tokens))
        else:
            final_chunks.append(chunk)

    return [c.strip() for c in final_chunks if len(c.strip()) >= MIN_CHUNK_CHARS]


def chunk_document(
    raw_text: str,
    company: str,
    ticker: str,
    source: str,
    source_type: str,
    published_at: str,
    url: str,
    max_tokens: int = MAX_TOKENS,
) -> List[Chunk]:
    """Chunk one document and attach metadata to each chunk."""
    texts = hybrid_chunk(raw_text, source_type=source_type, max_tokens=max_tokens)

    chunks = []
    for i, chunk_text in enumerate(texts):
        chunks.append(Chunk(
            text=chunk_text,
            company=company,
            ticker=ticker.upper() if ticker else "",
            source=source,
            source_type=source_type,
            published_at=str(published_at)[:10],
            url=str(url),
            chunk_index=i,
            chunk_total=len(texts),
        ))

    return chunks


def chunk_news_dataframe(df_news: pd.DataFrame, ticker: str = "") -> List[Chunk]:
    """Chunk cleaned news articles."""
    all_chunks = []

    for _, row in df_news.iterrows():
        chunks = chunk_document(
            raw_text=row.get("raw_text", ""),
            company=row.get("company", ""),
            ticker=ticker,
            source=row.get("source", ""),
            source_type="news",
            published_at=row.get("published_at", ""),
            url=row.get("url", ""),
        )
        all_chunks.extend(chunks)

    log.info(f"News chunking: {len(df_news)} articles -> {len(all_chunks)} chunks")
    return all_chunks


def chunk_sec_dataframe(df_sec: pd.DataFrame) -> List[Chunk]:
    """Chunk SEC filings, including 8-K earnings, Risk Factors, and MD&A."""
    all_chunks = []

    for _, row in df_sec.iterrows():
        source_type = str(row.get("section_type", "sec")).lower()

        if "earnings" in source_type:
            source_type = "earnings"
        elif "risk" in source_type:
            source_type = "risk"
        elif "m&a" in source_type or "md&a" in source_type:
            source_type = "mda"

        chunks = chunk_document(
            raw_text=row.get("text", ""),
            company=row.get("company", ""),
            ticker=row.get("ticker", ""),
            source=row.get("source", "SEC EDGAR"),
            source_type=source_type,
            published_at=row.get("filing_date", ""),
            url=row.get("url", ""),
        )
        all_chunks.extend(chunks)

    log.info(f"SEC chunking: {len(df_sec)} documents -> {len(all_chunks)} chunks")
    return all_chunks


def chunks_to_dataframe(chunks: List[Chunk]) -> pd.DataFrame:
    """Convert chunk objects into dataframe for RAG / FinBERT."""
    if not chunks:
        return pd.DataFrame()

    df_chunks = pd.DataFrame([{
        "company": c.company,
        "ticker": c.ticker,
        "source": c.source,
        "source_type": c.source_type,
        "published_at": c.published_at,
        "url": c.url,
        "chunk_index": c.chunk_index,
        "chunk_total": c.chunk_total,
        "text": c.text,
        "char_count": c.char_count,
        "token_count": c.token_count,
    } for c in chunks])

    df_chunks["section_type"] = df_chunks["source_type"].map({
        "risk": "Risk Factors",
        "mda": "MD&A",
        "earnings": "8-K earnings",
        "news": "news",
    }).fillna(df_chunks["source_type"])

    df_chunks["section_weight"] = df_chunks["section_type"].map({
        "MD&A": 1.5,
        "Risk Factors": 1.3,
        "8-K earnings": 1.0,
        "news": 0.8,
    }).fillna(1.0)

    df_chunks["filing_date"] = df_chunks["published_at"]

    return df_chunks

In [12]:
# =============================================================================
# 7. End-to-end preprocessing wrapper
# =============================================================================

def run_preprocessing(
    question: str,
    ticker: str = "",
    newsapi_key: Optional[str] = None,
    include_edgar: bool = False,
    company_name_for_edgar: Optional[str] = None,
    sec_identity: Optional[str] = None,
) -> Dict[str, pd.DataFrame]:
    """
    One clean function to run the full preprocessing stage.

    Returns a dictionary:
        {
            "df_news": clean article-level news dataframe,
            "df_debug": dropped article debug dataframe,
            "df_sec": optional SEC filings dataframe,
            "df_chunks": final chunk-level dataframe,
            "metadata": metadata dictionary
        }
    """
    df_news, df_debug, metadata = retrieve_clean_news(
        question=question,
        newsapi_key=newsapi_key,
    )

    company = metadata["company"]
    company_for_edgar = company_name_for_edgar or company

    all_chunks = []
    news_chunks = chunk_news_dataframe(df_news, ticker=ticker)
    all_chunks.extend(news_chunks)

    df_sec = pd.DataFrame()

    if include_edgar:
        if not ticker:
            raise ValueError("ticker is required when include_edgar=True.")
        if not sec_identity:
            raise ValueError("sec_identity is required when include_edgar=True.")

        df_earnings = get_earnings_filings(
            ticker=ticker,
            company_name=company_for_edgar,
            sec_identity=sec_identity,
        )

        df_10k = get_latest_10k_sections(
            ticker=ticker,
            company_name=company_for_edgar,
            sec_identity=sec_identity,
        )

        df_sec = pd.concat([df_earnings, df_10k], ignore_index=True)
        sec_chunks = chunk_sec_dataframe(df_sec)
        all_chunks.extend(sec_chunks)

    df_chunks = chunks_to_dataframe(all_chunks)

    metadata.update({
        "ticker": ticker,
        "include_edgar": include_edgar,
        "num_news_chunks": len(news_chunks),
        "num_sec_docs": len(df_sec),
        "num_total_chunks": len(df_chunks),
    })

    return {
        "df_news": df_news,
        "df_debug": df_debug,
        "df_sec": df_sec,
        "df_chunks": df_chunks,
        "metadata": metadata,
    }

In [13]:
# =============================================================================
# 8. Helpers for integration with rag_pipeline.py
# =============================================================================

def news_dataframe_to_rag_articles(df_news: pd.DataFrame) -> List[Dict[str, str]]:
    """
    Convert df_news into the article format expected by rag_pipeline.index_articles().

    rag_pipeline.index_articles() expects:
        [
            {"id": "...", "title": "...", "text": "..."}
        ]
    """
    articles = []

    for i, row in df_news.iterrows():
        text = str(row.get("raw_text", "")).strip()

        if len(text) < MIN_TEXT_LEN:
            continue

        articles.append({
            "id": f"news_{i}",
            "title": str(row.get("title", f"News Article {i}")),
            "text": text,
            "url": str(row.get("url", "")),
            "source": str(row.get("source", "")),
        })

    return articles


def chunks_dataframe_to_rag_articles(df_chunks: pd.DataFrame) -> List[Dict[str, str]]:
    """
    Alternative integration:
    Treat each already-created chunk as one RAG document.
    Use this only if you modify rag_pipeline so it does not chunk again.
    """
    articles = []

    for i, row in df_chunks.iterrows():
        text = str(row.get("text", "")).strip()

        if len(text) < MIN_CHUNK_CHARS:
            continue

        title = f"{row.get('company', '')} | {row.get('section_type', '')} | chunk {row.get('chunk_index', i)}"

        articles.append({
            "id": f"chunk_{i}",
            "title": title,
            "text": text,
            "url": str(row.get("url", "")),
            "source": str(row.get("source", "")),
        })

    return articles

In [59]:
%%writefile rag_pipeline.py

"""
Financial News RAG Pipeline
============================
Steps:
  1. Chunking        - split raw article text into overlapping chunks
  2. Embedding       - embed chunks via OpenAI text-embedding-3-small
  3. Vector DB       - store/load with ChromaDB (local, no server needed)
  4. Query           - embed the user query
  5. Similarity Search - retrieve top-k candidates from ChromaDB
  6. Reranking       - rerank with Cohere rerank API, keep top 5
  7. Prompt Engineering + LLM - send context + query to OpenAI GPT
"""
import os
import textwrap
from typing import List, Dict

# pip install openai chromadb cohere
import openai
import chromadb
import cohere

from openai import OpenAI
client = OpenAI()

# ---------------------------------------------------------------------------
# 0.  API Keys  (set as env vars or paste directly for homework)
# ---------------------------------------------------------------------------

#openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
#co            = cohere.Client(COHERE_API_KEY)
openai.api_key = os.getenv("OPENAI_API_KEY")
co = cohere.Client(os.getenv("COHERE_API_KEY"))

# ---------------------------------------------------------------------------
# 1.  Chunking
# ---------------------------------------------------------------------------
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Split text into word-level chunks with overlap.
    chunk_size : number of words per chunk
    overlap    : number of words shared between consecutive chunks
    """
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # slide forward with overlap
    return chunks


# ---------------------------------------------------------------------------
# 2.  Embedding  (OpenAI text-embedding-3-small, 1536-dim, very cheap)
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

local_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts):
    embeddings = local_model.encode(texts, convert_to_numpy=True)
    return embeddings.tolist()

# ---------------------------------------------------------------------------
# 3.  Vector Database  (ChromaDB, persisted locally in ./chroma_db/)
# ---------------------------------------------------------------------------
CHROMA_PATH       = "./chroma_db"
COLLECTION_NAME   = "financial_news"

def get_collection():
    """Return (or create) a persistent ChromaDB collection."""
    client     = chromadb.PersistentClient(path=CHROMA_PATH)
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},   # cosine similarity
    )
    return collection


def index_articles(articles: List[Dict[str, str]]):
    """
    Chunk, embed, and store a list of articles into ChromaDB.

    articles : list of {"id": "...", "title": "...", "text": "..."}
    """
    collection = get_collection()
    all_ids, all_docs, all_embeddings, all_metas = [], [], [], []

    for article in articles:
        chunks = chunk_text(article["text"])
        print(f"  Article '{article['title']}': {len(chunks)} chunks")

        embeddings = embed_texts(chunks)

        for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
            chunk_id = f"{article['id']}_chunk_{i}"
            all_ids.append(chunk_id)
            all_docs.append(chunk)
            all_embeddings.append(emb)
            all_metas.append({"article_id": article["id"],
                               "title":      article["title"],
                               "chunk_idx":  i})

    # Upsert so re-running doesn't duplicate
    collection.upsert(
        ids        = all_ids,
        documents  = all_docs,
        embeddings = all_embeddings,
        metadatas  = all_metas,
    )
    print(f"Indexed {len(all_ids)} chunks into ChromaDB.")


# ---------------------------------------------------------------------------
# 4 & 5.  Query + Similarity Search  (retrieve top-k candidates)
# ---------------------------------------------------------------------------
def similarity_search(query: str, top_k: int = 20) -> List[Dict]:
    """
    Embed the query and retrieve the top_k most similar chunks.
    Returns list of {"id", "document", "metadata", "distance"}.
    """
    collection     = get_collection()
    query_embedding = embed_texts([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    candidates = []
    for doc, meta, dist, cid in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
        results["ids"][0],
    ):
        candidates.append({
            "id":       cid,
            "document": doc,
            "metadata": meta,
            "distance": dist,   # cosine distance (lower = more similar)
        })
    return candidates


# ---------------------------------------------------------------------------
# 6.  Reranking  (Cohere rerank, keep top 5)
# ---------------------------------------------------------------------------
def rerank(query: str, candidates: List[Dict], top_n: int = 5) -> List[Dict]:
    """
    Use Cohere's rerank API to re-score candidates and return the best top_n.
    """
    docs = [c["document"] for c in candidates]

    response = co.rerank(
        model     = "rerank-english-v3.0",
        query     = query,
        documents = docs,
        top_n     = top_n,
    )

    reranked = []
    for result in response.results:
        candidate = candidates[result.index]
        candidate["rerank_score"] = result.relevance_score
        reranked.append(candidate)

    return reranked


# ---------------------------------------------------------------------------
# 7.  Prompt Engineering + LLM  (OpenAI GPT-4o-mini, cheap & fast)
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """You are a financial analyst assistant.
Answer the user's question based ONLY on the provided news context.
If the context does not contain enough information, say so clearly.
Be concise and factual."""

def build_prompt(query: str, context_chunks: List[Dict]) -> str:
    """Format retrieved chunks into a structured context block."""
    context_parts = []
    for i, chunk in enumerate(context_chunks, 1):
        title = chunk["metadata"].get("title", "Unknown")
        context_parts.append(
            f"[Source {i} – {title}]\n{chunk['document']}"
        )
    context_text = "\n\n".join(context_parts)
    return f"Context:\n{context_text}\n\nQuestion: {query}"


def ask_llm(query: str, context_chunks: List[Dict]) -> str:
    """
    Simple local answer generator.
    This avoids OpenAI API quota issues for demo purposes.
    """
    titles = []
    snippets = []

    for i, chunk in enumerate(context_chunks, 1):
        title = chunk["metadata"].get("title", "Unknown")
        text = chunk["document"][:500]

        titles.append(title)
        snippets.append(f"Source {i}: {title}\n{text}")

    answer = (
        f"Based on the retrieved recent news, the system found {len(context_chunks)} "
        f"relevant news chunks for the query: '{query}'.\n\n"
        f"The retrieved articles suggest that the current market narrative is mainly driven by:\n"
        f"- recent company-specific news,\n"
        f"- public discussion around Meta,\n"
        f"- and the sentiment extracted from the retrieved financial text.\n\n"
        f"Retrieved evidence:\n\n" + "\n\n".join(snippets)
    )

    return answer


# ---------------------------------------------------------------------------
# 8. Convert preprocessing dataframe to RAG input
# ---------------------------------------------------------------------------
def dataframe_to_articles(df, text_col="raw_text", title_col="title", url_col="url"):
    """
    Convert the cleaned dataframe from preprocessing3.ipynb into the format
    required by index_articles().
    """
    articles = []

    for i, row in df.iterrows():
        text = str(row.get(text_col, "")).strip()
        title = str(row.get(title_col, f"Article {i}")).strip()
        url = str(row.get(url_col, "")).strip()

        if len(text) < 200:
            continue

        articles.append({
            "id": f"article_{i}",
            "title": title,
            "text": text,
            "url": url
        })

    return articles


# ---------------------------------------------------------------------------
# 9. FinBERT Sentiment Scoring
# ---------------------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

FINBERT_MODEL_NAME = "ProsusAI/finbert"

finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL_NAME)
finbert_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL_NAME)
finbert_model.eval()

def score_sentiment_with_finbert(texts):
    """
    Score each retrieved chunk using FinBERT.
    Output labels: positive, negative, neutral.
    """
    results = []

    for text in texts:
        inputs = finbert_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = finbert_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1).numpy()[0]

        label_map = finbert_model.config.id2label

        score = {
            label_map[i].lower(): float(probs[i])
            for i in range(len(probs))
        }

        results.append(score)

    return results


def aggregate_sentiment(top_chunks):
    """
    Weighted average sentiment by Cohere rerank score.
    """
    texts = [chunk["document"] for chunk in top_chunks]
    scores = score_sentiment_with_finbert(texts)

    weights = np.array([
        chunk.get("rerank_score", 1.0)
        for chunk in top_chunks
    ])

    weights = weights / weights.sum()

    final_score = {
        "positive": 0.0,
        "negative": 0.0,
        "neutral": 0.0
    }

    for w, score in zip(weights, scores):
        for label in final_score:
            final_score[label] += w * score.get(label, 0.0)

    final_label = max(final_score, key=final_score.get)

    return {
        "final_label": final_label,
        "probabilities": final_score,
        "chunk_level_scores": scores
    }


# ---------------------------------------------------------------------------
# Full Pipeline (end-to-end)
# ---------------------------------------------------------------------------
def rag_query_with_sentiment(query: str, top_k_retrieve: int = 20, top_n_rerank: int = 5):
    """
    Full RAG pipeline:
    1. Similarity search
    2. Cohere rerank
    3. FinBERT sentiment scoring
    4. LLM answer generation
    """
    print(f"\n[Query] {query}")

    candidates = similarity_search(query, top_k=top_k_retrieve)
    print(f"[Retrieval] {len(candidates)} candidates retrieved")

    top_chunks = rerank(query, candidates, top_n=top_n_rerank)
    print(f"[Rerank] Top {len(top_chunks)} chunks selected")

    sentiment = aggregate_sentiment(top_chunks)
    print("\n[Sentiment]")
    print(sentiment)

    answer = ask_llm(query, top_chunks)
    print(f"\n[Answer]\n{answer}")

    return {
        "query": query,
        "answer": answer,
        "sentiment": sentiment,
        "sources": [
            {
                "title": chunk["metadata"].get("title", "Unknown"),
                "chunk": chunk["document"],
                "rerank_score": chunk.get("rerank_score")
            }
            for chunk in top_chunks
        ]
    }


# ---------------------------------------------------------------------------
# Demo / Smoke Test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # ---- Sample financial news articles (replace with your real corpus) ----
    sample_articles = [
        {
            "id":    "article_001",
            "title": "Fed Raises Rates by 25bps",
            "text":  """
                The Federal Reserve raised its benchmark interest rate by 25 basis points on Wednesday,
                bringing the federal funds rate to a target range of 5.25%–5.50%. Chair Jerome Powell
                signaled that further hikes remain possible if inflation does not cool sufficiently.
                Markets reacted with a modest sell-off in equities, while the US dollar strengthened
                against major peers. The 10-year Treasury yield climbed to 4.35%, its highest level
                since 2007. Analysts at Goldman Sachs noted that the move was widely anticipated and
                described it as a 'hawkish pause'. Consumer spending data released earlier in the week
                showed resilience, complicating the Fed's path toward a soft landing. Several regional
                bank stocks fell more than 2% on concerns about funding costs and net interest margins.
            """,
        },
        {
            "id":    "article_002",
            "title": "Apple Q3 Earnings Beat Expectations",
            "text":  """
                Apple Inc. reported third-quarter revenue of $81.8 billion, beating Wall Street estimates
                of $79.6 billion, driven by strong iPhone 15 sales and record Services revenue of $21.2
                billion. Net income rose 5% year-over-year to $19.9 billion. CEO Tim Cook highlighted
                growth in emerging markets, particularly India and Southeast Asia. The company announced
                a $90 billion share buyback program and raised its quarterly dividend by 4 cents to $0.24
                per share. Gross margins expanded to 44.5%, the highest in a decade. Analysts upgraded
                the stock to 'Buy', with price targets ranging from $200 to $230. Apple shares gained
                4.1% in after-hours trading following the earnings release.
            """,
        },
        {
            "id":    "article_003",
            "title": "Oil Prices Slump on Demand Fears",
            "text":  """
                Crude oil prices fell sharply on Thursday, with Brent crude dropping 3.2% to $78.40 per
                barrel and WTI declining to $74.10, as weak manufacturing data from China fueled concerns
                about global demand. OPEC+ sources indicated the group would maintain existing production
                cuts through the end of the year, but traders remained unconvinced this would offset the
                demand headwinds. Energy stocks in the S&P 500 shed 1.8% on average. Natural gas futures
                also fell 2.5% amid mild weather forecasts for the US northeast. Refining margins
                contracted to their lowest level since January. Analysts at Morgan Stanley reduced their
                year-end Brent forecast from $95 to $85 per barrel.
            """,
        },
    ]

    # ---- Index (only needs to run once; ChromaDB persists to disk) ---------
    print("=== Indexing Articles ===")
    index_articles(sample_articles)

    # ---- Query examples ----------------------------------------------------
    print("\n=== Running RAG Queries ===")

    rag_query("What did the Federal Reserve decide about interest rates?")
    rag_query("How did Apple perform in its latest earnings report?")
    rag_query("Why did oil prices fall and what is the outlook?")



Overwriting rag_pipeline.py


In [35]:
!pip install openai chromadb cohere transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.5/350.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [57]:
!pip install sentence-transformers

In [60]:
import os

os.environ["OPENAI_API_KEY"] = "YOUR_NEWSAPI_KEY"
os.environ["COHERE_API_KEY"] = "YOUR_NEWSAPI_KEY"

import importlib
import rag_pipeline
importlib.reload(rag_pipeline)

from rag_pipeline import (
    dataframe_to_articles,
    index_articles,
    rag_query_with_sentiment
)

articles = dataframe_to_articles(outputs["df_news"])

print("Number of articles:", len(articles))

index_articles(articles)

result = rag_query_with_sentiment(
    "What is happening with Meta lately?"
)

print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of articles: 12
  Article 'Meta is tracking employee keystrokes on Google, LinkedIn, Wikipedia as part of AI training initiative': 3 chunks
  Article 'David’s Bridal exec has a warning for every CEO obsessed with AI’s return-on-investment': 3 chunks
  Article 'Kenyan firm sacks more than 1,000 workers after losing Meta contract': 2 chunks
  Article 'Meta will start tracking employees’ screens and keystrokes to train AI tools': 2 chunks
  Article 'Greenhouse gases from data center boom could outpace entire nations': 10 chunks
  Article 'Threads is adding Live Chats to boost real-time engagement | TechCrunch': 3 chunks
  Article 'Meta will record employees' keystrokes and use it to train its AI models | TechCrunch': 1 chunks
  Article 'New Gas-Powered Data Centers Could Emit More Greenhouse Gases Than Entire Nations': 3 chunks
  Article 'SaySo is a new short-form video app that aims to restore users' trust in news | TechCrunch': 3 chunks
  Article 'Best Meta Glasses (2026): Ray-Ba

In [61]:
# =============================================================================
# 9. Example usage
# =============================================================================

if __name__ == "__main__":
    # In Colab, set your key like:
    import os
    os.environ["NEWSAPI_KEY"] = "43a329c2b8274822b8720bcf9f8f8db1"


    question = "What is happening with Meta lately?"

    outputs = run_preprocessing(
        question=question,
        ticker="META",
        include_edgar=False,
    )

    print("\n=== Metadata ===")
    print(outputs["metadata"])

    print("\n=== Clean News ===")
    print(outputs["df_news"][["company", "title", "source", "published_at", "char_count"]].head())

    print("\n=== Chunks ===")
    print(outputs["df_chunks"][["company", "ticker", "section_type", "source", "token_count", "text"]].head())

    # Integration with rag_pipeline.py:
    from rag_pipeline import dataframe_to_articles, index_articles, rag_query_with_sentiment
    articles = dataframe_to_articles(outputs["df_news"])
    index_articles(articles)
    result = rag_query_with_sentiment(question)
    print(result)


=== Metadata ===
{'question': 'What is happening with Meta lately?', 'company': 'Meta', 'days_back': 7, 'num_candidates': 38, 'num_kept': 13, 'num_dropped': 25, 'ticker': 'META', 'include_edgar': False, 'num_news_chunks': 38, 'num_sec_docs': 0, 'num_total_chunks': 38}

=== Clean News ===
  company                                              title  \
0    Meta  David’s Bridal exec has a warning for every CE...   
1    Meta  Meta is tracking employee keystrokes on Google...   
2    Meta  Kenyan firm sacks more than 1,000 workers afte...   
3    Meta  Meta will start tracking employees’ screens an...   
4    Meta  Greenhouse gases from data center boom could o...   

                  source              published_at  char_count  
0                Fortune 2026-04-22 19:10:30+00:00        4031  
1                   CNBC 2026-04-23 00:18:18+00:00        3853  
2  The Guardian Business 2026-04-17 16:59:04+00:00        2686  
3                Fortune 2026-04-21 18:53:46+00:00        2311  
